# Bab 17. Praktik Baik dan Reproduktibilitas

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import (LinearRegression,
    LogisticRegression, Ridge)
from sklearn.model_selection import (KFold, StratifiedKFold,
    train_test_split, cross_val_score, GridSearchCV,
    TimeSeriesSplit)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, roc_auc_score,
    f1_score)

from siapkan import kue_diperkaya

df = kue_diperkaya()

## 1. Benih menjamin pengulangan

In [ ]:
def jalankan(seed):
    r = np.random.default_rng(seed)
    return r.normal(size=5)

print(np.array_equal(jalankan(42), jalankan(42)))

Keluaran yang diharapkan:

```
True
```

## 2. Satu pemanggilan menggeser semuanya

In [ ]:
g = np.random.default_rng(0)
y1 = g.random(3)

g2 = np.random.default_rng(0)
g2.random(1)            # pemanggilan tambahan
y2 = g2.random(3)

print(np.array_equal(y1, y2))

Keluaran yang diharapkan:

```
False
```

## 3. Mencatat lingkungan

In [ ]:
import sys, numpy, pandas, sklearn

for m in [numpy, pandas, sklearn]:
    print(m.__name__, m.__version__)
print(sys.version.split()[0])

Keluaran yang diharapkan:

```
numpy 2.4.4
pandas 3.0.2
scikit-learn 1.8.0
3.12.3
```

## 4. Menyimpan model dengan lengkap

In [ ]:
import joblib

joblib.dump({
    "pipeline": pipa,
    "fitur": FITUR,
    "versi": {"sklearn": sklearn.__version__,
              "numpy": np.__version__,
              "pandas": pd.__version__},
    "dilatih_sampai": "2025-11-30",
}, "models/model.joblib")

muat = joblib.load("models/model.joblib")
p2 = muat["pipeline"].predict(uji[muat["fitur"]])
print(np.array_equal(p1, p2))

Keluaran yang diharapkan:

```
True
```

## 5. Asersi sebagai jaring pengaman

In [ ]:
def periksa(d):
    galat = []
    if d["jumlah"].min() < 1:
        galat.append("jumlah < 1")
    if d["jumlah"].isna().any():
        galat.append("ada nilai hilang")
    if not set(d["kanal"]) <= {"toko", "online", "reseller"}:
        galat.append("kanal asing")
    return galat

print(periksa(df_bersih))
print(periksa(df_rusak))

Keluaran yang diharapkan:

```
[]
['jumlah < 1', 'kanal asing']
```

## 6. Sidik jari hasil

In [ ]:
import hashlib

def sidik_jari(d):
    kunci = f"{len(d)}|{d.jumlah.sum()}|{d.jumlah.mean():.6f}"
    return hashlib.sha256(kunci.encode()).hexdigest()[:12]

print(sidik_jari(df))

Keluaran yang diharapkan:

```
5dad024b2d8b
```

## 7. Pencatatan minimal

In [ ]:
import csv, datetime

def catat(nama, params, skor, berkas="reports/catatan.csv"):
    with open(berkas, "a", newline="") as f:
        w = csv.writer(f)
        w.writerow([datetime.datetime.now().isoformat(),
                    nama, str(params), f"{skor:.4f}"])